# 4-QAM Classification With a Tiny MLP in MASE

This tutorial builds a very small MLP that classifies 4-QAM symbols from **I/Q** inputs.

- Input: two real values `(I, Q)`
- Output: four logits, one per 4-QAM symbol

Then we follow the MASE workflow style used in the other tutorials: baseline training, quantization, pruning, and post-transform fine-tuning.

## Workflow

1. Generate a synthetic noisy 4-QAM dataset
2. Train a tiny MLP baseline
3. Quantize with `quantize_transform_pass`
4. Fine-tune the quantized model (QAT-style)
5. Prune with `prune_transform_pass`
6. Fine-tune the pruned model
7. Save artifacts to `docs/source/modules/documentation/tutorials/classification-model`

In [ ]:
from pathlib import Path
import random

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from chop import MaseGraph
import chop.passes as passes

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
ARTIFACT_DIR = Path("docs/source/modules/documentation/tutorials/classification-model")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Device: {DEVICE}")
print(f"Artifact directory: {ARTIFACT_DIR.resolve()}")

In [ ]:
# 4-QAM constellation points: (-1,-1), (-1,+1), (+1,-1), (+1,+1)
CONSTELLATION = torch.tensor(
    [[-1.0, -1.0], [-1.0, 1.0], [1.0, -1.0], [1.0, 1.0]],
    dtype=torch.float32,
)

def make_4qam_split(samples_per_class: int, noise_std: float):
    xs = []
    ys = []
    for label, center in enumerate(CONSTELLATION):
        noise = torch.randn(samples_per_class, 2) * noise_std
        points = center.unsqueeze(0) + noise
        labels = torch.full((samples_per_class,), label, dtype=torch.long)
        xs.append(points)
        ys.append(labels)
    x = torch.cat(xs, dim=0)
    y = torch.cat(ys, dim=0)
    perm = torch.randperm(x.size(0))
    return x[perm], y[perm]

x_train, y_train = make_4qam_split(samples_per_class=2000, noise_std=0.30)
x_test, y_test = make_4qam_split(samples_per_class=600, noise_std=0.30)

train_loader = DataLoader(TensorDataset(x_train, y_train), batch_size=128, shuffle=True)
test_loader = DataLoader(TensorDataset(x_test, y_test), batch_size=512, shuffle=False)

print(f"Train set: {len(x_train)} samples")
print(f"Test set:  {len(x_test)} samples")

In [ ]:
class TinyQAMMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 16),
            nn.ReLU(),
            nn.Linear(16, 4),
        )

    def forward(self, x):
        return self.net(x)

def accuracy(model, loader, device):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            yb = yb.to(device)
            pred = model(xb).argmax(dim=1)
            correct += (pred == yb).sum().item()
            total += yb.numel()
    return correct / max(total, 1)

def train_model(model, loader, device, epochs=5, lr=1e-2):
    model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        for xb, yb in loader:
            xb = xb.to(device)
            yb = yb.to(device)
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * yb.size(0)
        avg_loss = running_loss / len(loader.dataset)
        val_acc = accuracy(model, test_loader, device)
        print(f"epoch={epoch:02d} loss={avg_loss:.4f} test_acc={val_acc:.4f}")

In [ ]:
baseline_model = TinyQAMMLP()
train_model(baseline_model, train_loader, DEVICE, epochs=12, lr=1e-2)
baseline_acc = accuracy(baseline_model, test_loader, DEVICE)
print(f"Baseline test accuracy: {baseline_acc:.4f}")

## Quantization

We now trace the trained model into a `MaseGraph`, add common metadata with a dummy input, and run integer quantization in `by="type"` mode (matching the tutorial/test style).

In [ ]:
mg = MaseGraph(model=baseline_model.cpu())
dummy_in = {"x": torch.randn(1, 2)}

mg, _ = passes.init_metadata_analysis_pass(mg)
mg, _ = passes.add_common_metadata_analysis_pass(
    mg,
    {"dummy_in": dummy_in, "add_value": False, "force_device_meta": False},
)

quantization_config = {
    "by": "type",
    "default": {"config": {"name": None}},
    "linear": {
        "config": {
            "name": "integer",
            "data_in_width": 8,
            "data_in_frac_width": 3,
            "weight_width": 8,
            "weight_frac_width": 3,
            "bias_width": 8,
            "bias_frac_width": 3,
        }
    },
    "relu": {"config": {"name": None}},
}

mg, _ = passes.quantize_transform_pass(mg, pass_args=quantization_config)
quant_model = mg.model.to(DEVICE)

quant_acc_pre_ft = accuracy(quant_model, test_loader, DEVICE)
print(f"Quantized accuracy before fine-tuning: {quant_acc_pre_ft:.4f}")

In [ ]:
# QAT-style fine-tuning: continue training after quantization
train_model(quant_model, train_loader, DEVICE, epochs=5, lr=3e-3)
quant_acc_post_ft = accuracy(quant_model, test_loader, DEVICE)
print(f"Quantized accuracy after fine-tuning: {quant_acc_post_ft:.4f}")

## Pruning

For pruning, we refresh metadata and enable `add_value=True` so pruning can access tensor values from metadata. Then we apply unstructured local L1 pruning to both weights and activations.

In [ ]:
mg = MaseGraph(model=quant_model.cpu())
mg, _ = passes.init_metadata_analysis_pass(mg)
mg, _ = passes.add_common_metadata_analysis_pass(
    mg,
    {"dummy_in": dummy_in, "add_value": True, "force_device_meta": False},
)

pruning_config = {
    "weight": {
        "scope": "local",
        "granularity": "elementwise",
        "method": "l1-norm",
        "sparsity": 0.35,
    },
    "activation": {
        "scope": "local",
        "granularity": "elementwise",
        "method": "l1-norm",
        "sparsity": 0.10,
    },
}

mg, _ = passes.prune_transform_pass(mg, pass_args=pruning_config)
pruned_model = mg.model.to(DEVICE)

pruned_acc_pre_ft = accuracy(pruned_model, test_loader, DEVICE)
print(f"Pruned accuracy before fine-tuning: {pruned_acc_pre_ft:.4f}")

In [ ]:
# Fine-tune after pruning to recover any accuracy drop
train_model(pruned_model, train_loader, DEVICE, epochs=5, lr=2e-3)
pruned_acc_post_ft = accuracy(pruned_model, test_loader, DEVICE)
print(f"Pruned accuracy after fine-tuning: {pruned_acc_post_ft:.4f}")

In [ ]:
torch.save(baseline_model.state_dict(), ARTIFACT_DIR / "baseline_fp32.pt")
torch.save(quant_model.state_dict(), ARTIFACT_DIR / "quantized_qat.pt")
torch.save(pruned_model.state_dict(), ARTIFACT_DIR / "pruned_qat.pt")

# Export the final MaseGraph checkpoint as in other tutorials
mg.export(str(ARTIFACT_DIR / "mase_graph_pruned_qat"))

summary = {
    "baseline_acc": float(baseline_acc),
    "quant_acc_pre_ft": float(quant_acc_pre_ft),
    "quant_acc_post_ft": float(quant_acc_post_ft),
    "pruned_acc_pre_ft": float(pruned_acc_pre_ft),
    "pruned_acc_post_ft": float(pruned_acc_post_ft),
}

print("Saved artifacts:")
for p in sorted(ARTIFACT_DIR.glob("*")):
    print(f" - {p.name}")

print("\nAccuracy summary:")
for k, v in summary.items():
    print(f"{k}: {v:.4f}")